# Lesson 4: OOP, Decorators & Modules

**Week 3 · Data Engineering Course**

---

This lesson covers how Python's class system works under the hood, how decorators are built from closures, how the module system resolves imports, and how to structure a data pipeline as a collection of well-defined, testable components.

**Sections:**
1. Python's Object System
2. Dunder Methods — Making Objects Behave
3. Closures and Decorators
4. Dataclasses
5. Python's Module and Import System
6. A Production-Style Pipeline Class

In [ ]:
import sys
import time
import functools
import logging
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Callable

print(f'Python {sys.version}')

---

## 4.1 Python's Object System

Everything in Python is an instance of a class. The class of an integer is `int`. The class of a function is `function`. The class of a class is `type`.

`type` is both a class and a function:
- `type(x)` returns the class of `x`
- `type(name, bases, attrs)` creates a new class at runtime

Each instance has a `__dict__` that stores its instance attributes. The class has its own `__dict__` that stores methods and class-level attributes. When you access `instance.attr`, Python searches instance `__dict__` first, then class `__dict__`, then base class `__dict__` up the hierarchy — the **Method Resolution Order (MRO)**.

In [ ]:
# Everything is an object
print(type(42))           # <class 'int'>
print(type('hello'))      # <class 'str'>
print(type(len))          # <class 'builtin_function_or_method'>
print(type(type))         # <class 'type'>  — type is its own type
print(type(int))          # <class 'type'>  — int is an instance of type

# type(name, bases, attrs) creates a class dynamically
MyRow = type('MyRow', (object,), {
    '__init__': lambda self, order_id, product: setattr(self, 'order_id', order_id) or setattr(self, 'product', product),
    '__repr__': lambda self: f'MyRow({self.order_id}, {self.product!r})'
})
row = MyRow(1001, 'Laptop')
print(row)                # MyRow(1001, 'Laptop')
print(type(row))          # <class '__main__.MyRow'>
print(type(row).__bases__) # (object,)

In [ ]:
# Instance __dict__ vs class __dict__
class DataSource:
    source_type = 'csv'    # class attribute — shared by all instances

    def __init__(self, path, encoding='utf-8'):
        self.path = path          # instance attribute — unique per instance
        self.encoding = encoding  # instance attribute

s1 = DataSource('/data/q1.csv')
s2 = DataSource('/data/q2.csv', encoding='latin-1')

print('s1.__dict__:', s1.__dict__)   # only instance attributes
print('s2.__dict__:', s2.__dict__)
print()
print('DataSource.__dict__ (partial):', {k: v for k, v in DataSource.__dict__.items() if not k.startswith('__')})
# source_type lives in the class, not the instance

# Attribute lookup: instance first, then class
print(f'\ns1.source_type: {s1.source_type}')   # found in class dict
print(f'DataSource.source_type: {DataSource.source_type}')

# Override at instance level: creates an instance attribute that shadows the class attribute
s1.source_type = 'parquet'
print(f's1.source_type after override: {s1.source_type}')   # instance
print(f's2.source_type unchanged: {s2.source_type}')         # still class attribute

In [ ]:
# Method Resolution Order (MRO): how Python finds methods in inheritance
class Base:
    def describe(self):
        return 'Base'

class Mixin:
    def extra(self):
        return 'Mixin'

class Child(Mixin, Base):  # multiple inheritance
    def describe(self):
        return f'Child -> {super().describe()}'

print('MRO:', [cls.__name__ for cls in Child.__mro__])
# [Child, Mixin, Base, object]
# Python searches left to right, depth first, with C3 linearisation

c = Child()
print(c.describe())   # Child -> Base
print(c.extra())      # Mixin

# __slots__: restrict instance attributes, save memory
class SlottedRow:
    __slots__ = ('order_id', 'product', 'total')  # no __dict__ created

    def __init__(self, order_id, product, total):
        self.order_id = order_id
        self.product = product
        self.total = total

r = SlottedRow(1001, 'Laptop', 1200)
print(f'\nSlottedRow order_id: {r.order_id}')
try:
    r.new_attr = 'x'   # AttributeError — __slots__ prevents arbitrary attributes
except AttributeError as e:
    print(f'Cannot add attribute: {e}')

import sys
regular = type('Regular', (), {'__init__': lambda self, x: setattr(self, 'x', x)})(42)
slotted = SlottedRow(1, 'x', 0)
print(f'Regular instance size: {sys.getsizeof(regular)} bytes')
print(f'Slotted instance size: {sys.getsizeof(slotted)} bytes')

---

## 4.2 Dunder Methods — Making Objects Behave

Dunder (double-underscore) methods let your class participate in Python's built-in operations. When you write `len(x)`, Python calls `x.__len__()`. When you write `x in y`, Python calls `y.__contains__(x)`. When you write `with x as f`, Python calls `x.__enter__()`.

For data engineering, the most useful dunders are:
- `__repr__`: readable representation in logs and REPLs
- `__len__`: makes your collection work with `len()`
- `__iter__` / `__next__`: makes your object iterable
- `__enter__` / `__exit__`: makes your object a context manager
- `__call__`: makes your object callable like a function

In [ ]:
# __repr__ and __str__
# repr: unambiguous, for developers. str: human-readable, for users.
class CleaningResult:
    def __init__(self, source, total, clean, dropped):
        self.source = source
        self.total = total
        self.clean = clean
        self.dropped = dropped

    def __repr__(self):
        return f'CleaningResult(source={self.source!r}, total={self.total}, clean={self.clean}, dropped={self.dropped})'

    def __str__(self):
        return f'{self.source}: {self.clean}/{self.total} rows clean, {self.dropped} dropped'

    def __eq__(self, other):
        if not isinstance(other, CleaningResult):
            return NotImplemented
        return (self.source, self.total, self.clean, self.dropped) == \
               (other.source, other.total, other.clean, other.dropped)

r1 = CleaningResult('sales_q1.csv', 26, 24, 2)
r2 = CleaningResult('sales_q1.csv', 26, 24, 2)
print(repr(r1))   # developer representation
print(str(r1))    # human representation
print(r1 == r2)   # True — uses __eq__

In [ ]:
# __len__, __iter__, __getitem__: making a class behave like a container
import csv
from pathlib import Path

class CSVDataset:
    '''Iterable, indexable wrapper around a list of CSV rows.'''

    def __init__(self, path: Path):
        self.path = path
        self._rows = []
        with open(path, newline='', encoding='utf-8') as f:
            self._rows = list(csv.DictReader(f))

    def __len__(self):
        return len(self._rows)

    def __iter__(self):
        return iter(self._rows)

    def __getitem__(self, index):
        return self._rows[index]

    def __repr__(self):
        return f'CSVDataset({self.path.name!r}, {len(self)} rows)'

RAW = Path('../data/raw')
ds = CSVDataset(RAW / 'sales_q1.csv')

print(ds)              # __repr__
print(len(ds))         # __len__
print(ds[0])           # __getitem__
print(ds[-1])          # negative indexing

for i, row in enumerate(ds):   # __iter__
    if i >= 2:
        break
    print(row['order_id'], row['product'])

In [ ]:
# __enter__ and __exit__: context manager
# __call__: callable object

class PipelineStep:
    '''A named transformation step that times itself and acts as a context manager.'''

    def __init__(self, name: str, func: Callable):
        self.name = name
        self.func = func
        self.elapsed_ms = None

    def __call__(self, data):
        '''Apply the step to data.'''
        start = time.perf_counter()
        result = self.func(data)
        self.elapsed_ms = (time.perf_counter() - start) * 1000
        return result

    def __enter__(self):
        print(f'[START] {self.name}')
        self._start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        elapsed = (time.perf_counter() - self._start) * 1000
        if exc_type is None:
            print(f'[END]   {self.name}: {elapsed:.1f} ms')
        else:
            print(f'[FAIL]  {self.name}: {elapsed:.1f} ms  ({exc_type.__name__}: {exc_val})')
        return False

    def __repr__(self):
        elapsed_str = f'{self.elapsed_ms:.1f} ms' if self.elapsed_ms else 'not run'
        return f'PipelineStep({self.name!r}, {elapsed_str})'

# Use as a callable
strip_step = PipelineStep('strip_whitespace', lambda rows: [{k: v.strip() if isinstance(v, str) else v for k, v in row.items()} for row in rows])

with PipelineStep('file_read', lambda _: None) as ctx:
    import csv
    with open(RAW / 'sales_q1.csv', newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))

result = strip_step(rows)
print(strip_step)

---

## 4.3 Closures and Decorators

A **closure** is a function that remembers the variables from the enclosing scope where it was defined, even after that scope has exited.

A **decorator** is a function that takes a function as input, wraps it, and returns the wrapper. The `@decorator` syntax is just shorthand for `func = decorator(func)`.

Decorators are used everywhere in data engineering: for timing steps, retrying on transient failures, logging inputs and outputs, validating results, and caching expensive computations.

In [ ]:
# Closures: what they are and why they work
def make_counter(start=0):
    count = start
    def increment(by=1):
        nonlocal count      # tells Python to use the enclosing scope's count
        count += by
        return count
    return increment

counter = make_counter()
print(counter())    # 1
print(counter())    # 2
print(counter(5))   # 7
print(counter.__closure__[0].cell_contents)  # 7 — the closure cell holds the state

# Each call to make_counter() creates a SEPARATE closure
counter_a = make_counter()
counter_b = make_counter(100)
print(f'\na: {counter_a()}')   # 1
print(f'b: {counter_b()}')   # 101  — independent

In [ ]:
# The simplest decorator: a function that wraps another function
def timer(func):
    @functools.wraps(func)   # copies __name__, __doc__ from func to wrapper
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = (time.perf_counter() - start) * 1000
        print(f'{func.__name__}: {elapsed:.2f} ms')
        return result
    return wrapper

@timer
def read_all_csv(folder: Path) -> list:
    '''Read all CSV files in folder. Returns list of dicts.'''
    all_rows = []
    for path in sorted(folder.glob('*.csv')):
        with open(path, newline='', encoding='utf-8') as f:
            all_rows.extend(csv.DictReader(f))
    return all_rows

rows = read_all_csv(RAW)
print(f'Total rows: {len(rows)}')
print(f'Function name preserved: {read_all_csv.__name__}')   # read_all_csv (not wrapper)
print(f'Docstring preserved: {read_all_csv.__doc__}')

In [ ]:
# Parametrised decorator: a decorator that itself takes arguments
# Requires one extra level of nesting

def retry(max_attempts: int = 3, delay_seconds: float = 1.0, exceptions=(Exception,)):
    '''Retry a function up to max_attempts times on specified exceptions.'''
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if attempt == max_attempts:
                        print(f'{func.__name__}: failed after {max_attempts} attempts')
                        raise
                    print(f'{func.__name__}: attempt {attempt} failed ({e}), retrying in {delay_seconds}s...')
                    time.sleep(delay_seconds)
        return wrapper
    return decorator

# Use: retries up to 3 times, 0.1s between attempts, on IOError only
call_count = 0

@retry(max_attempts=3, delay_seconds=0.1, exceptions=(IOError,))
def flaky_read(path: Path) -> str:
    global call_count
    call_count += 1
    if call_count < 3:      # simulate failure on first two attempts
        raise IOError(f'Simulated I/O error (attempt {call_count})')
    return path.read_text(encoding='utf-8')[:50]

try:
    result = flaky_read(RAW / 'sales_q1.csv')
    print(f'Success on attempt {call_count}: {result!r}')
except IOError as e:
    print(f'All attempts failed: {e}')

In [ ]:
# @property: computed attributes that look like attributes

class Pipeline:
    def __init__(self, name: str):
        self.name = name
        self._steps = []
        self._results = []

    def add_step(self, func):
        self._steps.append(func)
        return self   # enables method chaining: p.add_step(a).add_step(b)

    @property
    def step_count(self):
        return len(self._steps)

    @property
    def has_results(self):
        return bool(self._results)

    @classmethod
    def from_list(cls, name: str, steps: list):
        '''Alternative constructor: create a Pipeline from a list of functions.'''
        p = cls(name)
        for step in steps:
            p.add_step(step)
        return p

    @staticmethod
    def validate_step(func):
        '''Check that a function is callable. Static: does not need self or cls.'''
        if not callable(func):
            raise TypeError(f'Expected callable, got {type(func).__name__}')
        return True

p = Pipeline('sales_cleaner')
p.add_step(str.strip).add_step(str.lower)
print(f'step_count: {p.step_count}')   # property: no () needed
print(f'has_results: {p.has_results}')

Pipeline.validate_step(str.strip)      # static method: called on the class
print(Pipeline.from_list('test', [str.strip, str.upper]))  # classmethod alternate constructor

---

## 4.4 Dataclasses

`@dataclass` is a decorator that generates `__init__`, `__repr__`, and `__eq__` from a class's type annotations. For configuration objects and result containers, it is almost always the right choice over writing a manual class.

What `@dataclass` generates for free:
- `__init__`: with one parameter per annotated field
- `__repr__`: showing all field names and values
- `__eq__`: comparing all fields

Optional extras: `order=True` generates `__lt__`/`__gt__`; `frozen=True` makes instances immutable (hashable); `slots=True` adds `__slots__` for memory efficiency.

In [ ]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class PipelineConfig:
    name: str
    input_dir: Path
    output_dir: Path
    encoding: str = 'utf-8'
    chunk_size: Optional[int] = None
    category_typos: dict = field(default_factory=dict)
    # field(default_factory=...) creates a new dict for each instance
    # NEVER write category_typos: dict = {} — that is the mutable default trap!

    def __post_init__(self):
        '''Validation runs after __init__.'''
        if not self.input_dir.exists():
            raise FileNotFoundError(f'Input directory not found: {self.input_dir}')
        self.output_dir.mkdir(parents=True, exist_ok=True)

cfg = PipelineConfig(
    name='week3_pipeline',
    input_dir=Path('../data/raw'),
    output_dir=Path('../data/clean'),
    category_typos={'Electrnics': 'Electronics'}
)

print(repr(cfg))    # __repr__ generated automatically
print(f'name: {cfg.name}')
print(f'chunk_size: {cfg.chunk_size}')  # None

In [ ]:
# frozen=True: immutable dataclass — safe to use as a dict key or set member
@dataclass(frozen=True)
class ColumnSpec:
    name: str
    dtype: str
    nullable: bool = False

    def __post_init__(self):
        valid_dtypes = {'str', 'int', 'float', 'date', 'bool'}
        if self.dtype not in valid_dtypes:
            raise ValueError(f'Invalid dtype {self.dtype!r}. Expected one of {valid_dtypes}')

SCHEMA = [
    ColumnSpec('order_id',   'int',   False),
    ColumnSpec('date',       'date',  False),
    ColumnSpec('product',    'str',   False),
    ColumnSpec('category',   'str',   False),
    ColumnSpec('quantity',   'int',   False),
    ColumnSpec('unit_price', 'float', False),
    ColumnSpec('total',      'float', True),
    ColumnSpec('sales_rep',  'str',   True),
]

# frozen dataclasses are hashable and can be used in sets
spec_set = {s.name for s in SCHEMA}
print(f'Schema columns: {spec_set}')
print(SCHEMA[0])

try:
    SCHEMA[0].name = 'changed'   # TypeError: frozen
except Exception as e:
    print(f'\nCannot modify frozen: {e}')

try:
    ColumnSpec('x', 'unknown_type')   # validation in __post_init__
except ValueError as e:
    print(f'Validation caught: {e}')

---

## 4.5 Python's Module and Import System

When you write `import csv`, Python:
1. Looks in `sys.modules` (the cache). If `csv` is already there, returns it immediately.
2. Searches `sys.path` — a list of directories. First match wins.
3. Compiles the file to bytecode (`.pyc`), caches it in `__pycache__`.
4. Executes the module code. All top-level statements run once.
5. Stores the module object in `sys.modules`.

Second `import csv` returns the cached module — module code runs only once.

For packages (directories), Python looks for `__init__.py`. Its contents are executed when the package is first imported, and define what `from package import X` can expose.

In [ ]:
import sys

# sys.modules: the import cache
print('csv in sys.modules:', 'csv' in sys.modules)
import csv
print('csv in sys.modules:', 'csv' in sys.modules)
print(f'sys.modules["csv"]: {sys.modules["csv"]}')

# Second import does NOT re-run the module
# This is why module-level expensive operations (DB connections, model loads)
# only happen once even when multiple files import the same module

# sys.path: where Python looks for modules
print('\nFirst 5 entries in sys.path:')
for p in sys.path[:5]:
    print(f'  {p}')

In [ ]:
# Writing a module file and importing it
from pathlib import Path
import importlib.util, sys

module_src = '''
"""Reusable cleaning functions for the Week 3 pipeline."""
from datetime import datetime
import re

# Compiled at import time — not re-compiled on every call
_DATE_PATTERNS = [
    (\'%Y-%m-%d\', re.compile(r\'\\\\d{4}-\\\\d{2}-\\\\d{2}\')),
    (\'%Y/%m/%d\', re.compile(r\'\\\\d{4}/\\\\d{2}/\\\\d{2}\')),
    (\'%d/%m/%Y\', re.compile(r\'\\\\d{2}/\\\\d{2}/\\\\d{4}\')),
    (\'%b %d %Y\', re.compile(r\'[A-Za-z]+ \\\\d+ \\\\d{4}\')),
]

def parse_date(s):
    """Parse a date string in any known format; return YYYY-MM-DD or None."""
    s = str(s).strip()
    for fmt, pattern in _DATE_PATTERNS:
        if pattern.fullmatch(s):
            try:
                return datetime.strptime(s, fmt).strftime(\' %Y-%m-%d\').strip()
            except ValueError:
                pass
    return None

def clean_price(value):
    """Remove currency symbols; return float or None."""
    try:
        return float(str(value).replace(\'$\', \'\').replace(\',\', \'\').strip())
    except (ValueError, TypeError):
        return None

__all__ = [\'parse_date\', \'clean_price\']
'''

mod_path = Path('temp_cleaners.py')
mod_path.write_text(module_src.replace("\''", "'"), encoding='utf-8')
print('Module written.')
mod_path.unlink()  # clean up the temp file right away

# In a real project, you would:
# 1. Create week3/cleaners/__init__.py with __all__ and imports
# 2. Import with: from cleaners import parse_date, clean_price

In [ ]:
# __all__ controls what `from module import *` exports
# It also signals to readers what the module's public API is

# Demonstrate with the csv module itself
print('csv.__all__:', csv.__all__)
# These are the names you get with `from csv import *`
# Everything else is internal

---

## 4.6 A Production-Style Pipeline Class

All of the concepts from this lesson come together in a pipeline class that:
- Uses `@dataclass` for configuration
- Uses `@timer` for per-step timing
- Implements `__enter__` / `__exit__` so the pipeline can be used as a context manager
- Uses structured `logging` rather than `print`
- Uses `@retry` on steps that may encounter transient failures
- Validates output row count before writing

In [ ]:
import csv
import logging
import time
import functools
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Callable, Iterator

# Configure structured logging (goes to stderr, not stdout)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(name)s  %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('pipeline')

# --- Configuration ---
@dataclass(frozen=True)
class PipelineConfig:
    name: str
    input_dir: Path
    output_path: Path
    encoding: str = 'utf-8'
    typo_map: dict = field(default_factory=dict)
    min_rows: int = 1    # raise if output has fewer rows than this

# --- Decorators ---
def step_timer(func):
    @functools.wraps(func)
    def wrapper(self, *args, **kwargs):
        log.info('START %s', func.__name__)
        t0 = time.perf_counter()
        result = func(self, *args, **kwargs)
        elapsed = (time.perf_counter() - t0) * 1000
        log.info('END   %s: %.1f ms', func.__name__, elapsed)
        return result
    return wrapper

def validate_output(min_rows: int = 1):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(self, rows, *args, **kwargs):
            result = func(self, rows, *args, **kwargs)
            if len(result) < min_rows:
                raise ValueError(f'{func.__name__} returned {len(result)} rows, expected >= {min_rows}')
            return result
        return wrapper
    return decorator

print('Classes and decorators defined.')

In [ ]:
class SalesPipeline:
    '''End-to-end cleaning pipeline for quarterly sales CSVs.'''

    def __init__(self, config: PipelineConfig):
        self.config = config
        self._rows = []
        self._start_time = None

    def __enter__(self):
        self._start_time = time.perf_counter()
        log.info('Pipeline %r started', self.config.name)
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        elapsed = (time.perf_counter() - self._start_time) * 1000
        if exc_type is None:
            log.info('Pipeline %r finished in %.1f ms, %d rows written',
                     self.config.name, elapsed, len(self._rows))
        else:
            log.error('Pipeline %r FAILED after %.1f ms: %s: %s',
                      self.config.name, elapsed, exc_type.__name__, exc_val)
        return False  # do not suppress exceptions

    @step_timer
    def read(self) -> 'SalesPipeline':
        '''Read all CSV files from input_dir.'''
        rows = []
        for path in sorted(self.config.input_dir.glob('*.csv')):
            with open(path, newline='', encoding=self.config.encoding) as f:
                file_rows = list(csv.DictReader(f))
                log.info('  read %d rows from %s', len(file_rows), path.name)
                rows.extend(file_rows)
        self._rows = rows
        return self

    @step_timer
    @validate_output(min_rows=50)
    def clean(self, rows: list) -> list:
        '''Clean rows: dedup, strip, normalise category, validate quantity.'''
        seen = set()
        result = []
        for row in rows:
            oid = row.get('order_id', '')
            if oid in seen:
                continue
            seen.add(oid)
            row['category'] = self.config.typo_map.get(
                row['category'].strip().title(),
                row['category'].strip().title()
            )
            row['product'] = row['product'].strip()
            try:
                qty = float(row['quantity'])
            except (ValueError, TypeError):
                continue
            if qty <= 0:
                continue
            row['quantity'] = int(qty)
            result.append(row)
        return result

    @step_timer
    def write(self, rows: list) -> None:
        '''Write cleaned rows to output_path atomically.'''
        out = self.config.output_path
        out.parent.mkdir(parents=True, exist_ok=True)
        tmp = out.with_suffix('.tmp')
        fieldnames = list(rows[0].keys()) if rows else []
        with open(tmp, 'w', newline='', encoding=self.config.encoding) as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(rows)
        tmp.rename(out)
        self._rows = rows

    def run(self) -> 'SalesPipeline':
        '''Execute the full pipeline.'''
        self.read()
        cleaned = self.clean(self._rows)
        self.write(cleaned)
        return self

print('SalesPipeline class defined.')

In [ ]:
# Run the pipeline
config = PipelineConfig(
    name='sales_2023',
    input_dir=Path('../data/raw'),
    output_path=Path('../data/clean/sales_2023.csv'),
    typo_map={
        'Electrnics': 'Electronics',
        'Home & Kithen': 'Home & Kitchen',
    },
    min_rows=50
)

with SalesPipeline(config) as pipeline:
    pipeline.run()

print(f'\nOutput exists: {config.output_path.exists()}')
with open(config.output_path, newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
print(f'Output rows: {len(rows)}')

In [ ]:
# Demonstrate that __exit__ handles exceptions cleanly
bad_config = PipelineConfig(
    name='will_fail',
    input_dir=Path('../data/raw'),
    output_path=Path('../data/clean/fail_output.csv'),
    min_rows=1000   # impossible threshold — triggers validate_output
)

try:
    with SalesPipeline(bad_config) as pipeline:
        pipeline.run()  # validate_output will raise ValueError
except ValueError as e:
    print(f'\nCaught outside with block: {e}')
    print('(The with block logged the failure and re-raised — __exit__ returned False)')

---

## Summary

| Concept | What to remember |
|---------|------------------|
| Object model | Every value is an object. `type(x)` returns its class. `id(x)` returns its memory address. Attribute lookup searches instance dict → class dict → base class dicts (MRO). |
| `__dict__` and `__slots__` | Instance attributes live in `__dict__`. `__slots__` removes `__dict__`, restricts allowed attributes, and reduces memory by 20–50 bytes per instance. |
| Dunder methods | `__repr__` for debugging, `__len__`/`__iter__`/`__getitem__` for container behaviour, `__enter__`/`__exit__` for context managers, `__call__` for callable objects. |
| Closures | A nested function captures variables from the enclosing scope. `nonlocal` lets you reassign them. Use closures to build pre-configured functions. |
| Decorators | A decorator is a function that wraps another function. Always use `@functools.wraps` to preserve the wrapped function's metadata. Parametrised decorators need one extra nesting level. |
| `@property` / `@classmethod` / `@staticmethod` | Property: computed attribute. Classmethod: alternative constructor, receives `cls` not `self`. Staticmethod: utility function that belongs in the class namespace but needs neither. |
| `@dataclass` | Generates `__init__`, `__repr__`, `__eq__`. Use `field(default_factory=...)` for mutable defaults. Use `frozen=True` for immutable config objects. |
| Module system | Modules run once and are cached in `sys.modules`. `sys.path` controls where Python looks. Top-level code in a module runs at import time — avoid expensive operations at the top level. |
| Pipeline class | Combine `@dataclass` config, `@step_timer` decorator, `__enter__`/`__exit__`, and structured `logging` to build a pipeline that is observable, testable, and safe to run in production. |